# Fake News Detection - Phase 9: 'Onion Or Not' Bias Correction

This version uses **'Real Weird News' (Not The Onion)** to fix the bias against absurd true stories.
1.  **Load OnionOrNot**: Contains 'The Onion' (Satire) and 'Not The Onion' (Real Weird News).
2.  **Filter**: Keep only Label 0 (Not The Onion). These are true stories that sound fake.
3.  **Inject**: Add 2,000 of these samples to teach the model that 'Weird' != 'Fake'.
4.  **Train**: Combined with WELFake + LIAR + Snopes.

## Instructions
1.  **Upload Files**: `best_model_state.bin`, `WELFake_Dataset.csv`, `train.tsv`, `valid.tsv`, `test.tsv`, `snopeswithsum.csv`, `OnionOrNot.csv`.
2.  **Run All**

In [ ]:
# 1. Install Dependencies
!pip install transformers torch pandas numpy matplotlib seaborn wordcloud scikit-learn sentencepiece accelerate

In [ ]:
# 2. Mount Drive & Define Paths
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/fake_news_detection'
MODEL_PATH = os.path.join(PROJECT_PATH, 'best_model_state.bin')
DATA_PATH_WELFAKE = os.path.join(PROJECT_PATH, 'WELFake_Dataset.csv')
DATA_PATH_LIAR_TRAIN = os.path.join(PROJECT_PATH, 'train.tsv')
DATA_PATH_LIAR_VALID = os.path.join(PROJECT_PATH, 'valid.tsv')
DATA_PATH_LIAR_TEST = os.path.join(PROJECT_PATH, 'test.tsv')
DATA_PATH_SNOPES = os.path.join(PROJECT_PATH, 'snopeswithsum.csv')
DATA_PATH_ONION = os.path.join(PROJECT_PATH, 'OnionOrNot.csv')
OUTPUT_PATH = os.path.join(PROJECT_PATH, 'best_model_state_scientific.bin')

if not os.path.exists(PROJECT_PATH):
    os.makedirs(PROJECT_PATH, exist_ok=True)
    print(f"⚠️ Created {PROJECT_PATH}. Please upload your files there!")
else:
    print(f"✅ Project folder found at: {PROJECT_PATH}")

In [ ]:
# 3. Preprocessing Function
import re
import string

def clean_text(text):
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

In [ ]:
# 4. Load & Clean Datasets
import pandas as pd
from sklearn.model_selection import train_test_split

# --- LIAR ---
def load_liar_manual(path):
    if not path or not os.path.exists(path):
        return pd.DataFrame(columns=['text', 'label'])
    try:
        df = pd.read_csv(path, sep='\t', header=None, usecols=[1, 2], names=['label_text', 'text'])
        label_map = {'false': 1, 'pants-fire': 1, 'barely-true': 1, 'true': 0, 'mostly-true': 0, 'half-true': 0}
        df['label'] = df['label_text'].map(label_map)
        df = df.dropna(subset=['label'])
        df['label'] = df['label'].astype(int)
        return df[['text', 'label']]
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return pd.DataFrame(columns=['text', 'label'])

df_liar = pd.concat([load_liar_manual(DATA_PATH_LIAR_TRAIN), load_liar_manual(DATA_PATH_LIAR_VALID), load_liar_manual(DATA_PATH_LIAR_TEST)])

# --- Snopes (Verified Labels) ---
def load_snopes(path):
    if not path or not os.path.exists(path):
        return pd.DataFrame(columns=['text', 'label']), pd.DataFrame(columns=['text', 'label'])
    try:
        print("Loading Snopes...")
        df = pd.read_csv(path)
        drop_cols = ['rate', "what's true", "what's false", "what's unknown", 'summary', 'origin', 'question', 'comment']
        # Verified Label Mapping
        label_map = {
            # Real
            'True': 0, 'Mostly True': 0, 'Correct Attribution': 0,
            # Fake
            'False': 1, 'Mostly False': 1, 'Miscaptioned': 1, 'Unproven': 1, 
            'Mixture': 1, 'Legend': 1, 'Lost Legend': 1, 'Scam': 1, 
            'Labeled Satire': 1, 'Misattributed': 1, 'Outdated': 1
        }
        df['label'] = df['rate'].map(label_map)
        df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
        df = df.dropna(subset=['label'])
        df['label'] = df['label'].astype(int)
        df = df.rename(columns={'claim': 'text'})
        train, test = train_test_split(df[['text', 'label']], test_size=0.2, random_state=42)
        return train, test
    except Exception as e:
        print(f"Error reading Snopes: {e}")
        return pd.DataFrame(columns=['text', 'label']), pd.DataFrame(columns=['text', 'label'])

df_snopes_train, df_snopes_test = load_snopes(DATA_PATH_SNOPES)

# --- WELFake ---
if os.path.exists(DATA_PATH_WELFAKE):
    df_welfake = pd.read_csv(DATA_PATH_WELFAKE)
    df_welfake_subset = df_welfake.sample(n=20000) 
else:
    df_welfake_subset = pd.DataFrame(columns=['text', 'label'])

# --- ONION OR NOT (Phase 9) ---
def load_onion(path):
    if not path or not os.path.exists(path):
        return pd.DataFrame(columns=['text', 'label'])
    try:
        print("Loading OnionOrNot...")
        df = pd.read_csv(path)
        # Label 0: Not The Onion (Real Weird News)
        # Label 1: The Onion (Satire/Fake)
        # Filter for Label 0 ONLY (We want Real Weird News)
        df_real_weird = df[df['label'] == 0].copy()
        
        # Sample 2000 rows (Enough to teach concept, not overwhelm)
        if len(df_real_weird) > 2000:
            df_real_weird = df_real_weird.sample(n=2000, random_state=42)
            
        print(f"Loaded {len(df_real_weird)} 'Real Weird' news samples.")
        return df_real_weird[['text', 'label']] # Label is already 0 (Real)
    except Exception as e:
        print(f"Error reading OnionOrNot: {e}")
        return pd.DataFrame(columns=['text', 'label'])

df_onion_real = load_onion(DATA_PATH_ONION)

In [ ]:
# --- Combine All Datasets ---
print("\nCombining All Datasets...")
# Note: df_snopes_train is used directly (no algo augmentation)
df_combined = pd.concat([df_welfake_subset[['text', 'label']], df_liar, df_snopes_train, df_onion_real])
df_combined = df_combined.sample(frac=1).reset_index(drop=True)

# Save Scientific Dataset
COMBINED_DATA_PATH = os.path.join(PROJECT_PATH, 'FINAL_SCIENTIFIC_TRAINING_SET.csv')
df_combined.to_csv(COMBINED_DATA_PATH, index=False)
print(f"✅ Saved Scientific Dataset to: {COMBINED_DATA_PATH } | Total Size: {len(df_combined)}")

# Apply Cleaning
print("Applying text cleaning to combined dataset...")
df_combined['text'] = df_combined['text'].apply(clean_text)

In [ ]:
# 5. Train with Standard Loss 
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertModel
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_combined['text'].values, df_combined['label'].values, test_size=0.1, random_state=42
)

class HybridBertBiLSTM(nn.Module):
    def __init__(self, n_classes):
        super(HybridBertBiLSTM, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.lstm = nn.LSTM(input_size=768, hidden_size=128, num_layers=1, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(128 * 2, n_classes)
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        lstm_out, _ = self.lstm(outputs.last_hidden_state)
        out = self.drop(torch.mean(lstm_out, dim=1))
        return self.out(out)

class SyntheticDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, item):
        text = str(self.texts[item])
        encoding = self.tokenizer.encode_plus(text, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'].flatten(), 'labels': torch.tensor(self.labels[item], dtype=torch.long)}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = HybridBertBiLSTM(n_classes=2).to(DEVICE)

train_loader = DataLoader(SyntheticDataset(train_texts, train_labels, tokenizer), batch_size=16, shuffle=True)
val_loader = DataLoader(SyntheticDataset(val_texts, val_labels, tokenizer), batch_size=16)
optimizer = AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.CrossEntropyLoss().to(DEVICE)

print("Starting Phase 9 Training...")
model.train()
for epoch in range(3):
    epoch_loss = 0
    correct_predictions = 0
    total_examples = 0
    for d in train_loader:
        input_ids, attention_mask, targets = d['input_ids'].to(DEVICE), d['attention_mask'].to(DEVICE), d['labels'].to(DEVICE)
        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, targets)
        
        _, preds = torch.max(outputs, dim=1)
        correct_predictions += torch.sum(preds == targets)
        total_examples += targets.size(0)
        epoch_loss += loss.item()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
    epoch_acc = correct_predictions.double() / total_examples
    print(f"Epoch {epoch+1} Complete | Loss: {epoch_loss/len(train_loader):.4f} | Acc: {epoch_acc:.4f}")

torch.save(model.state_dict(), OUTPUT_PATH)
print(f"Saved model to {OUTPUT_PATH}")

# Verification
if len(df_snopes_test) > 0:
    print("\n--- Evaluating on Held-Out Snopes Test Set (Phase 9) ---")
    snopes_loader = DataLoader(SyntheticDataset(df_snopes_test['text'].values, df_snopes_test['label'].values, tokenizer), batch_size=16)
    all_preds, all_labels = [], []
    with torch.no_grad():
        for d in snopes_loader:
            outputs = model(d['input_ids'].to(DEVICE), d['attention_mask'].to(DEVICE))
            all_preds.extend(torch.max(outputs, 1)[1].cpu().numpy())
            all_labels.extend(d['labels'].cpu().numpy())
    print(classification_report(all_labels, all_preds, target_names=['Real', 'Fake']))
    
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
    plt.title('Confusion Matrix (Snopes Test Set)')
    plt.show()